# 01d – Independent evaluation set: 30 randomly sampled areas

The test set of rounds 1–3 has only six areas, and it was already looked at during development.
This notebook builds a **new, untouched evaluation set** to estimate how the final model works in places
nobody chose by hand.

**Sampling (documented, repeatable):**
1. A random generator with a fixed seed draws candidate points inside a rough box around each continent.
2. Candidates are rejected if they are mostly water (WorldCover), closer than 50 km to any of the 67 project areas,
   to Mannheim or to another accepted candidate, have no Sentinel-1 track with images in all four months,
   or less than 90 % radar coverage.
3. For each continent, the **first five** valid candidates in the drawn order are kept (30 areas in total).
4. Every candidate that was looked at and the reason for its rejection is saved (`holdout/config/checked_v4_holdout.json`).
   This also measures **data availability** for the candidates checked in this procedure.

**Re-running this notebook** gives exactly the same areas and patches (fixed seed, same rules). If saved holdout files
already exist, they are **not overwritten**: the new result is compared with them and the notebook stops if anything
differs. Saved radar checks are reused only if they were made with the same settings (year, months, area size,
coverage rule).

**Labels:** ESA WorldCover 2021, exactly as in training. The result is therefore *agreement with WorldCover on unseen
random areas*, not proven real-world correctness.

**Rules:** these areas are never used for training or for choosing a model. The final model is frozen
(notebooks 02d and 02c) **before** this set is evaluated (notebook 02e).

The files are saved outside `data/` (in `holdout/`), so the training notebooks can never pick them up.

# 1. Setup

In [1]:
import sys
import subprocess

# packages for the data download (Earth Engine, rasterio, ...), same versions as in requirements.txt
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("packages ready")

packages ready


In [2]:
from pathlib import Path
import json
import math

import numpy as np
import requests
import rasterio
import rasterio.errors
import ee

PROJECT_ID = "sentinel1-lulc-project-512618"

# Only asks for a new login when the saved token is missing. Clear this cell's output before sharing.
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready:", ee.Number(1).getInfo())


Successfully saved authorization token.
Earth Engine ready: 1


# 2. Areas that must be avoided

All 67 project areas (round 1: 33 + the dropped Ontario_forest, round 2: 34) and the case-study city Mannheim.

In [3]:
YEAR = 2021
MONTHS = [1, 4, 7, 10]
BUFFER_METRES = 7700
PLAN_VERSION = "v4_holdout"
SPLIT = "holdout"

project_areas = [
    # round 1 (name, lon, lat)
    ("Abisko", 18.830, 68.350), ("Lyon", 4.840, 45.760), ("Madrid", -3.700, 40.420), ("Warsaw", 21.010, 52.230),
    ("Prague", 14.440, 50.080), ("Toulouse", 1.444, 43.605), ("Ludhiana_rural", 75.650, 30.900),
    ("Sundarbans", 89.550, 22.100), ("Pune", 73.857, 18.520), ("Delhi", 77.210, 28.610),
    ("Bangkok_edge", 100.750, 13.850), ("Hanoi_edge", 105.900, 21.100), ("Nairobi", 36.820, -1.290),
    ("Marrakech_edge", -8.150, 31.650), ("Cairo", 31.240, 30.040), ("Lagos", 3.380, 6.520),
    ("Arusha_edge", 36.750, -3.450), ("Pretoria_edge", 28.350, -25.750), ("Sacramento_rural", -121.650, 38.550),
    ("Ontario_forest", -78.500, 45.500), ("Chicago", -87.750, 41.850), ("Mexico_City", -99.130, 19.430),
    ("Austin_edge", -97.600, 30.250), ("Portland_edge", -122.680, 45.520), ("Santarem_edge", -54.800, -2.550),
    ("Rosario_rural", -60.900, -33.000), ("Sao_Paulo", -46.630, -23.550), ("Curitiba_edge", -49.150, -25.550),
    ("Cordoba_edge", -64.050, -31.450), ("Wagga_rural", 147.200, -35.100), ("Darwin_wetlands", 131.300, -12.650),
    ("Melbourne", 144.960, -37.810), ("Perth_edge", 115.950, -31.800), ("Christchurch", 172.650, -43.500),
    # round 2
    ("Tabernas_desert", -2.390, 37.050), ("Riyadh_edge", 46.450, 24.750), ("Ouarzazate", -6.900, 30.920),
    ("Phoenix_edge", -112.300, 33.350), ("Antofagasta", -70.400, -23.650), ("Alice_Springs", 133.880, -23.700),
    ("Camargue", 4.550, 43.550), ("Danube_delta", 29.300, 45.150), ("Poyang_lake", 116.250, 29.000),
    ("Okavango", 22.900, -19.300), ("Everglades_edge", -80.600, 25.800), ("Pantanal", -57.000, -17.600),
    ("Dublin", -6.260, 53.350), ("Po_valley", 10.500, 45.100), ("Jyvaskyla_forest", 25.700, 62.000),
    ("Andalusia_farmland", -4.800, 37.550), ("Jakarta", 106.850, -6.200), ("Dhaka", 90.410, 23.810),
    ("Tehran", 51.400, 35.700), ("Astana_steppe", 71.450, 51.150), ("Mekong_delta", 105.800, 10.000),
    ("Addis_Ababa", 38.740, 9.030), ("Kano_rural", 8.500, 12.000), ("Kinshasa", 15.310, -4.320),
    ("Nile_delta", 31.000, 30.800), ("Iowa_farmland", -93.600, 42.000), ("Alberta_prairie", -113.500, 52.000),
    ("Guadalajara", -103.350, 20.670), ("Bogota", -74.100, 4.650), ("Lima", -77.050, -12.050),
    ("Manaus_edge", -60.000, -3.050), ("Buenos_Aires", -58.450, -34.600), ("Brisbane", 153.020, -27.470),
    ("Auckland", 174.760, -36.850),
    # case study
    ("Mannheim", 8.466, 49.487),
]
avoid = [{"name": n, "lon": lon, "lat": lat} for n, lon, lat in project_areas]
print(len(avoid), "areas to avoid")


def distance_km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, [a["lat"], a["lon"], b["lat"], b["lon"]])
    h = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * math.asin(math.sqrt(h))


MIN_DISTANCE_KM = 50

69 areas to avoid


# 3. Draw the candidates

Each continent is described by one or two rough boxes (longitude from–to, latitude from–to). The boxes do not
overlap. They are only used to spread the areas over the world: small parts of neighbouring regions fall into a box
(for example the Red Sea coast of Sudan into the Asia box, Panama into the South America box), and some remote parts
are not covered (Siberia, Alaska, the Pacific islands).

Points are drawn **evenly by area**: a box is picked in proportion to its surface on the globe, and the latitude is
drawn so that high latitudes are not over-represented (on a map, one degree of longitude near the poles is much
shorter than at the equator). Points in the sea are removed in the next step.

Every continent has its own random generator (seed 2026 + continent number). Drawing more candidates later only adds
new points at the end; the first ones stay the same.

In [4]:
SEED = 2026
PER_CONTINENT = 5           # areas kept per continent
DRAWS_PER_CONTINENT = 80    # candidates drawn per continent (many are in the sea or have no radar data)

boxes = {
    "Europe":        [(-10, 34, 37.5, 70), (34, 60, 55, 70)],
    "Asia":          [(34, 145, 12, 55), (95, 141, -9, 12)],
    "Africa":        [(-17.5, 34, -35, 37.5), (34, 51.5, -27, 12)],
    "North America": [(-125, -60, 15, 60)],
    "South America": [(-82, -34, -56, 12.5)],
    "Oceania":       [(113, 154, -44, -10), (166, 179, -48, -34)],
}


def box_area(box):
    lon_min, lon_max, lat_min, lat_max = box
    return (lon_max - lon_min) * (math.sin(math.radians(lat_max)) - math.sin(math.radians(lat_min)))


def draw_point(rng, continent_boxes):
    weights = np.array([box_area(b) for b in continent_boxes])
    lon_min, lon_max, lat_min, lat_max = continent_boxes[rng.choice(len(continent_boxes), p=weights / weights.sum())]
    lon = rng.uniform(lon_min, lon_max)
    lat = math.degrees(math.asin(rng.uniform(math.sin(math.radians(lat_min)), math.sin(math.radians(lat_max)))))
    return round(float(lon), 4), round(float(lat), 4)


candidates = []
for number, (continent, continent_boxes) in enumerate(boxes.items()):
    rng = np.random.default_rng([SEED, number])
    for i in range(DRAWS_PER_CONTINENT):
        lon, lat = draw_point(rng, continent_boxes)
        candidates.append({"name": f"{continent.replace(' ', '_')}_{i:03d}", "continent": continent,
                           "lon": lon, "lat": lat, "split": SPLIT})
print(len(candidates), "candidates drawn")

480 candidates drawn


## Land check for all candidates at once

Share of land in WorldCover (everything except water and "no data") on a 100 m grid, in one request.

In [5]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
land = worldcover.neq(80).unmask(0).rename("land")      # sea and lakes -> 0, outside WorldCover -> 0

features = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([c["lon"], c["lat"]]).buffer(BUFFER_METRES).bounds(), {"name": c["name"]})
    for c in candidates
])
land_share = land.reduceRegions(collection=features, reducer=ee.Reducer.mean(), scale=100, tileScale=4).getInfo()
land_by_name = {f["properties"]["name"]: f["properties"].get("mean", 0) or 0 for f in land_share["features"]}

for c in candidates:
    c["land_share"] = round(100 * land_by_name.get(c["name"], 0), 1)
print("candidates with at least 50 % land:", sum(c["land_share"] >= 50 for c in candidates))

candidates with at least 50 % land: 271


# 4. Check the candidates in the drawn order

Per continent, candidates are checked one after the other until five are accepted.
Checks: at least 50 % land, distance rules, one Sentinel-1 track with images in all four months,
at least 90 % radar coverage in every month.

In [6]:
config_folder = Path("holdout/config")
config_folder.mkdir(parents=True, exist_ok=True)
log_path = config_folder / f"candidates_{PLAN_VERSION}.json"
MIN_COVERAGE = 90


def add_track(image):
    direction = ee.String(image.get("orbitProperties_pass"))
    orbit = ee.Number(image.get("relativeOrbitNumber_start")).format("%d")
    return image.set("track", direction.cat("_").cat(orbit))


def sentinel1_year(region):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )


def check_radar(area):
    # same rules as notebooks 01 and 01b: one common track, then coverage of that track
    region = ee.Geometry.Point([area["lon"], area["lat"]]).buffer(BUFFER_METRES).bounds()
    collection = sentinel1_year(region).map(add_track)
    queries = {}
    for month in MONTHS:
        start = ee.Date.fromYMD(YEAR, month, 1)
        queries[str(month)] = collection.filterDate(start, start.advance(1, "month")).aggregate_histogram("track")
    counts = ee.Dictionary(queries).getInfo()

    common = set(counts[str(MONTHS[0])])
    for month in MONTHS[1:]:
        common.intersection_update(counts[str(month)])
    if not common:
        return "no_common_track", {}

    best = max(sorted(common), key=lambda t: (min(counts[str(m)][t] for m in MONTHS),
                                              sum(counts[str(m)][t] for m in MONTHS)))
    direction, orbit = best.rsplit("_", 1)
    info = {"direction": direction, "orbit": int(orbit), "scene_counts": [int(counts[str(m)][best]) for m in MONTHS]}

    zone = min(60, max(1, math.floor((area["lon"] + 180) / 6) + 1))
    crs = f"EPSG:{(32600 if area['lat'] >= 0 else 32700) + zone}"
    track = (sentinel1_year(region)
             .filter(ee.Filter.eq("orbitProperties_pass", direction))
             .filter(ee.Filter.eq("relativeOrbitNumber_start", int(orbit)))
             .select(["VV", "VH"]))
    bands = []
    for month in MONTHS:
        start = ee.Date.fromYMD(YEAR, month, 1)
        bands.append(track.filterDate(start, start.advance(1, "month")).count()
                     .unmask(0, sameFootprint=False).reduce(ee.Reducer.min()).gt(0).rename(f"month_{month:02d}"))
    coverage = ee.Image.cat(*bands).reduceRegion(reducer=ee.Reducer.mean(), geometry=region, crs=crs,
                                                  scale=10, maxPixels=10_000_000, tileScale=4).getInfo()
    info["coverage_percent"] = {k: round(100 * v, 2) for k, v in coverage.items()}
    if min(info["coverage_percent"].values()) < MIN_COVERAGE:
        return "low_coverage", info
    return "accepted", info

In [7]:
# radar results are saved, so a restarted run does not ask Earth Engine again;
# a saved result is only reused if it was made for the same point with the same settings.
# The quick rules (land, distances) are checked again every time, in the drawn order.
RADAR_SETTINGS = {"year": YEAR, "months": MONTHS, "buffer_metres": BUFFER_METRES, "min_coverage": MIN_COVERAGE,
                  "sentinel1": "COPERNICUS/S1_GRD, IW, 10 m, VV + VH, one common track"}

radar_log = json.loads(log_path.read_text()) if log_path.exists() else {}
accepted, checked = [], []

for continent in boxes:
    kept = 0
    for c in [c for c in candidates if c["continent"] == continent]:
        if kept == PER_CONTINENT:
            break
        if c["land_share"] < 50:
            entry = {**c, "status": "mostly_water"}
        elif any(distance_km(c, a) < MIN_DISTANCE_KM for a in avoid):
            entry = {**c, "status": "too_close_to_project_area"}
        elif any(distance_km(c, a) < MIN_DISTANCE_KM for a in accepted):
            entry = {**c, "status": "too_close_to_other_candidate"}
        else:
            entry = radar_log.get(c["name"])
            reusable = (entry is not None and (entry["lon"], entry["lat"]) == (c["lon"], c["lat"])
                        and entry.get("settings") == RADAR_SETTINGS and entry["status"] != "query_error")
            if not reusable:
                try:
                    status, info = check_radar(c)
                    entry = {**c, **info, "status": status, "settings": RADAR_SETTINGS}
                except Exception as error:
                    entry = {**c, "status": "query_error", "error": str(error)[:300]}
                radar_log[c["name"]] = entry
                log_path.write_text(json.dumps(radar_log, indent=2))
            print(f"{c['name']:<20} land {c['land_share']:5.1f}%  -> {entry['status']}", flush=True)
        checked.append(entry)
        if entry["status"] == "accepted":
            accepted.append(entry)
            kept += 1
    print(f"== {continent}: {kept} accepted\n")
    if kept < PER_CONTINENT:
        print(f"   WARNING: only {kept} – increase DRAWS_PER_CONTINENT and run this section again\n")

print("accepted areas:", len(accepted))
assert len(accepted) == PER_CONTINENT * len(boxes), "not enough accepted areas"

# the complete record of this selection: every candidate that was looked at, with its reason
checked_path = config_folder / f"checked_{PLAN_VERSION}.json"
checked_path.write_text(json.dumps({"settings": {**RADAR_SETTINGS, "seed": SEED, "per_continent": PER_CONTINENT,
                                                 "draws_per_continent": DRAWS_PER_CONTINENT, "boxes": boxes,
                                                 "min_land_percent": 50, "min_distance_km": MIN_DISTANCE_KM},
                                    "candidates": checked}, indent=2))
print("saved", checked_path)

# the accepted areas must be the same as in a previous run (they define the holdout set used in 02e)
IDENTITY = ["name", "continent", "lon", "lat", "direction", "orbit"]
accepted_path = config_folder / f"accepted_{PLAN_VERSION}.json"
if accepted_path.exists():
    previous = json.loads(accepted_path.read_text())
    if [{k: a.get(k) for k in IDENTITY} for a in previous] != [{k: a.get(k) for k in IDENTITY} for a in accepted]:
        raise RuntimeError(f"The accepted areas differ from the saved {accepted_path}; the saved file is kept.")
    print("same 30 accepted areas (names, places, tracks) as in the saved file")
    for old, new in zip(previous, accepted):
        for key in ["scene_counts", "coverage_percent"]:
            if old.get(key) != new.get(key):
                print(f"  note: {new['name']} {key} now {new.get(key)}, saved {old.get(key)} (Earth Engine archive changed)")
else:
    accepted_path.write_text(json.dumps(accepted, indent=2))

Europe_000           land  97.7%  -> accepted
Europe_001           land  81.0%  -> accepted
Europe_002           land  98.9%  -> accepted
Europe_003           land  89.4%  -> accepted
Europe_004           land  99.3%  -> accepted
== Europe: 5 accepted

Asia_002             land  99.9%  -> accepted
Asia_003             land  97.4%  -> accepted
Asia_005             land 100.0%  -> accepted
Asia_007             land  98.7%  -> accepted
Asia_009             land 100.0%  -> low_coverage
Asia_011             land  99.8%  -> accepted
== Asia: 5 accepted

Africa_003           land 100.0%  -> accepted
Africa_004           land 100.0%  -> accepted
Africa_006           land 100.0%  -> accepted
Africa_007           land 100.0%  -> accepted
Africa_008           land 100.0%  -> accepted
== Africa: 5 accepted

North_America_002    land  99.9%  -> accepted
North_America_003    land  88.8%  -> accepted
North_America_004    land  99.7%  -> accepted
North_America_007    land 100.0%  -> accepted
North_Ame

## Data availability: why candidates were rejected

Only candidates that were looked at are counted (checking stops after five accepted per continent).

In [8]:
reasons = ["accepted", "mostly_water", "too_close_to_project_area", "too_close_to_other_candidate",
           "no_common_track", "low_coverage", "query_error"]
print(f"{'continent':<15}" + "".join(f"{r[:14]:>16}" for r in reasons))
for continent in boxes:
    entries = [e for e in checked if e["continent"] == continent]
    print(f"{continent:<15}" + "".join(f"{sum(e['status'] == r for e in entries):16d}" for r in reasons))

on_land = [e for e in checked if e["status"] not in ("mostly_water", "too_close_to_project_area",
                                                    "too_close_to_other_candidate")]
if on_land:
    ok = sum(e["status"] == "accepted" for e in on_land)
    print(f"\nradar data usable for {ok} of {len(on_land)} land candidates that were checked ({100 * ok / len(on_land):.0f} %)")

print("\naccepted:")
for a in accepted:
    print(f"  {a['name']:<20} lon {a['lon']:9.4f}  lat {a['lat']:8.4f}  {a['direction']:<10} {a['orbit']:4d}  scenes {a['scene_counts']}")

continent              accepted    mostly_water  too_close_to_p  too_close_to_o  no_common_trac    low_coverage     query_error
Europe                        5               0               0               0               0               0               0
Asia                          5               6               0               0               0               1               0
Africa                        5               4               0               0               0               0               0
North America                 5               6               0               0               0               0               0
South America                 5               7               0               1               0               1               0
Oceania                       5               3               0               0               0               2               0

radar data usable for 30 of 34 land candidates that were checked (88 %)

accepted:
  Europe_000        

# 5. Export radar and labels

Same code as notebooks 01 / 01b (one track per area, original radar grid, files checked against `info.json`).

In [9]:
year, months, buffer_metres = YEAR, MONTHS, BUFFER_METRES
selected_areas = accepted
saved = {"checks": {a["name"]: {"status": "complete"} for a in accepted}}

In [10]:
NODATA = -9999
PIXEL_SIZE = 10
raw_folder = Path("holdout/raw") / PLAN_VERSION
raw_folder.mkdir(parents=True, exist_ok=True)

# settings that define an area's files; if any of them changes, the files must be downloaded again
IDENTITY_KEYS = ["lon", "lat", "direction", "orbit", "crs", "width", "height"]

export_areas = [
    area for area in selected_areas
    if saved["checks"].get(area["name"], {}).get("status") == "complete"
]
print("Areas to export:", len(export_areas))


def track_collection(area, region):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(ee.Filter.eq("orbitProperties_pass", area["direction"]))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", area["orbit"]))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
    )


def snapped_grid(region, collection):
    # Same as the pilot: snap the area to the original radar pixel grid
    radar_projection = ee.Image(collection.first()).select("VV").projection().getInfo()

    crs = radar_projection["crs"]
    origin_x = radar_projection["transform"][2]
    origin_y = radar_projection["transform"][5]

    bounds = region.bounds(maxError=1, proj=ee.Projection(crs)).coordinates().getInfo()[0]
    xs = [point[0] for point in bounds]
    ys = [point[1] for point in bounds]

    left_col = math.floor((min(xs) - origin_x) / PIXEL_SIZE)
    right_col = math.ceil((max(xs) - origin_x) / PIXEL_SIZE)
    top_row = math.floor((origin_y - max(ys)) / PIXEL_SIZE)
    bottom_row = math.ceil((origin_y - min(ys)) / PIXEL_SIZE)

    width = right_col - left_col
    height = bottom_row - top_row
    left = origin_x + left_col * PIXEL_SIZE
    top = origin_y - top_row * PIXEL_SIZE

    return crs, [PIXEL_SIZE, 0, left, 0, -PIXEL_SIZE, top], width, height


def download_tif(image, path, crs, grid_transform, width, height):
    url = image.getDownloadURL({
        "crs": crs,
        "crs_transform": grid_transform,
        "dimensions": [width, height],
        "format": "GEO_TIFF"
    })
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    path.write_bytes(response.content)

    with rasterio.open(path, "r+") as dataset:
        dataset.nodata = NODATA


def check_raster(path, info, band_count):
    # returns "ok" or the reason why the file does not match info.json
    if not path.exists():
        return "missing"
    try:
        with rasterio.open(path) as src:
            if src.crs is None or src.crs.to_string() != info["crs"]:
                return "wrong coordinate system"
            if (src.width, src.height) != (info["width"], info["height"]):
                return "wrong size"
            # rtol=0: the default relative tolerance would accept shifts of ~50 m on UTM coordinates
            if not np.allclose(tuple(src.transform)[:6], info["transform"], rtol=0, atol=1e-6):
                return "wrong pixel grid"
            if src.count != band_count:
                return f"{src.count} bands instead of {band_count}"
    except rasterio.errors.RasterioIOError:
        return "cannot be opened"
    return "ok"


def same_settings(old_info, new_info):
    return (all(old_info.get(key) == new_info[key] for key in IDENTITY_KEYS)
            and np.allclose(old_info.get("transform", [0] * 6), new_info["transform"], rtol=0, atol=1e-6))

Areas to export: 30


In [11]:
for index, area in enumerate(export_areas, start=1):
    name = area["name"]
    area_folder = raw_folder / name
    area_folder.mkdir(parents=True, exist_ok=True)

    info_path = area_folder / "info.json"
    label_path = area_folder / f"worldcover_{year}.tif"
    month_paths = {month: area_folder / f"s1_{year}_{month:02d}.tif" for month in months}

    print(f"[{index}/{len(export_areas)}] {name}", flush=True)

    try:
        region = ee.Geometry.Point([area["lon"], area["lat"]]).buffer(buffer_metres).bounds()
        collection = track_collection(area, region)
        crs, grid_transform, width, height = snapped_grid(region, collection)

        info = {
            **area, "year": year, "months": months, "buffer_metres": buffer_metres,
            "crs": crs, "transform": grid_transform, "width": width, "height": height
        }

        # files from other settings (or without info.json) cannot be trusted
        old_info = json.loads(info_path.read_text()) if info_path.exists() else None
        if old_info is None or not same_settings(old_info, info):
            for old_file in area_folder.glob("*.tif"):
                old_file.unlink()
                print("  deleted outdated", old_file.name)
        info_path.write_text(json.dumps(info, indent=2), encoding="utf-8")

        for month, path in month_paths.items():
            if check_raster(path, info, 2) == "ok":
                continue
            start = ee.Date.fromYMD(year, month, 1)
            monthly = collection.filterDate(start, start.advance(1, "month")).median()
            download_tif(monthly.toFloat().unmask(NODATA, sameFootprint=False),
                         path, crs, grid_transform, width, height)
            status = check_raster(path, info, 2)
            if status != "ok":
                raise ValueError(f"{path.name}: {status}")
            print(f"  month {month:02d} saved")

        if check_raster(label_path, info, 1) != "ok":
            download_tif(worldcover.toInt16().unmask(NODATA, sameFootprint=False),
                         label_path, crs, grid_transform, width, height)
            status = check_raster(label_path, info, 1)
            if status != "ok":
                raise ValueError(f"{label_path.name}: {status}")
            print("  labels saved")

        print("  all files ok")

    except Exception as error:
        print("  Export failed:", error)

[1/30] Europe_000
  all files ok
[2/30] Europe_001
  all files ok
[3/30] Europe_002
  all files ok
[4/30] Europe_003
  all files ok
[5/30] Europe_004
  all files ok
[6/30] Asia_002
  all files ok
[7/30] Asia_003
  all files ok
[8/30] Asia_005
  all files ok
[9/30] Asia_007
  all files ok
[10/30] Asia_011
  all files ok
[11/30] Africa_003
  all files ok
[12/30] Africa_004
  all files ok
[13/30] Africa_006
  all files ok
[14/30] Africa_007
  all files ok
[15/30] Africa_008
  all files ok
[16/30] North_America_002
  all files ok
[17/30] North_America_003
  all files ok
[18/30] North_America_004
  all files ok
[19/30] North_America_007
  all files ok
[20/30] North_America_010
  all files ok
[21/30] South_America_004
  all files ok
[22/30] South_America_006
  all files ok
[23/30] South_America_008
  all files ok
[24/30] South_America_010
  all files ok
[25/30] South_America_013
  all files ok
[26/30] Oceania_000
  all files ok
[27/30] Oceania_002
  all files ok
[28/30] Oceania_003
  all fil

# 6. Patches

Same classes and rules as in training (7 classes, snow and no data = 255; patches with missing radar or less than
90 % labelled pixels are skipped). All patches belong to the split `holdout`. The number of kept and skipped patches
per area is saved in `holdout/config/patches_v4_holdout.json`, and every accepted area must have patches.

In [12]:
PATCH_SIZE = 256
IGNORE = 255

CLASSES = ["Trees (incl. mangroves)", "Grassland / shrubland", "Cropland", "Built-up",
           "Bare / sparse vegetation", "Permanent water", "Herbaceous wetland"]

# 70 Snow and ice is left out on purpose -> IGNORE
REMAP = {10: 0, 95: 0, 30: 1, 20: 1, 100: 1, 40: 2, 50: 3, 60: 4, 80: 5, 90: 6}

lookup = np.full(101, IGNORE, dtype=np.uint8)
for code, new_class in REMAP.items():
    lookup[code] = new_class


def to_classes(codes):
    classes = np.full(codes.shape, IGNORE, dtype=np.uint8)
    known = (codes >= 0) & (codes <= 100)
    classes[known] = lookup[codes[known]]
    return classes

In [13]:
patch_folder = Path(f"holdout/patches_{PLAN_VERSION}")
patch_folder.mkdir(parents=True, exist_ok=True)

images, masks, meta = [], [], []
patch_record = {}
for area in export_areas:
    name = area["name"]
    area_folder = raw_folder / name
    info_path = area_folder / "info.json"
    label_path = area_folder / f"worldcover_{year}.tif"
    month_paths = {month: area_folder / f"s1_{year}_{month:02d}.tif" for month in months}

    if not info_path.exists():
        print(f"{name:<20} skipped: no info.json")
        continue
    info = json.loads(info_path.read_text())

    problems = {label_path.name: check_raster(label_path, info, 1)}
    for path in month_paths.values():
        problems[path.name] = check_raster(path, info, 2)
    problems = {file: status for file, status in problems.items() if status != "ok"}
    if problems:
        print(f"{name:<20} skipped:", problems)
        continue

    with rasterio.open(label_path) as src:
        labels = to_classes(src.read(1))

    kept, skipped_radar, skipped_labels = 0, 0, 0
    for month, path in month_paths.items():
        with rasterio.open(path) as src:
            radar = src.read()
        height, width = labels.shape
        for row in range(0, height - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, width - PATCH_SIZE + 1, PATCH_SIZE):
                x = radar[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE]
                y = labels[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
                if (x == NODATA).any() or not np.isfinite(x).all():
                    skipped_radar += 1
                    continue
                if (y != IGNORE).mean() < 0.9:
                    skipped_labels += 1
                    continue
                images.append(np.moveaxis(x, 0, -1).astype(np.float16))
                masks.append(y)
                meta.append(f"{name},{area['continent']},{month},{row},{col}")
                kept += 1
    patch_record[name] = {"kept": kept, "skipped_missing_radar": skipped_radar, "skipped_few_labels": skipped_labels}
    print(f"{name:<20} patches: {kept:4d}   skipped: {skipped_radar} missing radar, {skipped_labels} too few labels")

missing_areas = [a["name"] for a in accepted if patch_record.get(a["name"], {}).get("kept", 0) == 0]
assert not missing_areas, f"accepted areas without patches: {missing_areas}"

x = np.stack(images)
y = np.stack(masks)
meta_text = "area,continent,month,row,col\n" + "\n".join(meta)
class_record = json.loads(json.dumps({
    "classes": CLASSES,
    "worldcover_to_class": {str(code): new for code, new in REMAP.items()},
    "ignore": IGNORE,
    "out_of_scope": {"70": "Snow and ice"},
    "plan_version": PLAN_VERSION,
    "sampling": {"seed": SEED, "per_continent": PER_CONTINENT, "boxes": boxes},
}))
files = {"x": patch_folder / f"{SPLIT}_x.npy", "y": patch_folder / f"{SPLIT}_y.npy",
         "meta": patch_folder / f"{SPLIT}_meta.csv", "classes": patch_folder / "classes.json"}

if all(path.exists() for path in files.values()):
    # a holdout set already exists (it was used in 02e): never overwrite it, only compare
    same = {
        "x": np.array_equal(np.load(files["x"], mmap_mode="r"), x),
        "y": np.array_equal(np.load(files["y"], mmap_mode="r"), y),
        "meta": files["meta"].read_text(encoding="utf-8") == meta_text,
        "classes": json.loads(files["classes"].read_text()) == class_record,
    }
    if not all(same.values()):
        raise RuntimeError(f"The new patches differ from the saved holdout set: {same}. The saved files are kept.")
    print("\nidentical to the saved holdout set (radar, labels, patch list and classes)")
else:
    np.save(files["x"], x)
    np.save(files["y"], y)
    files["meta"].write_text(meta_text, encoding="utf-8")
    files["classes"].write_text(json.dumps(class_record, indent=2))
    print("\nholdout set saved")

(config_folder / f"patches_{PLAN_VERSION}.json").write_text(json.dumps(
    {"areas": len(patch_record), "patches": int(len(x)), "per_area": patch_record}, indent=2))

labelled = y[y != IGNORE]
print(f"\nholdout: x {x.shape}, areas {len(set(m.split(',')[0] for m in meta))}")
for i, name in enumerate(CLASSES):
    print(f"  {name:<26}{100 * np.mean(labelled == i):6.2f}%")

Europe_000           patches:  144   skipped: 0 missing radar, 0 too few labels
Europe_001           patches:  144   skipped: 0 missing radar, 0 too few labels
Europe_002           patches:  144   skipped: 0 missing radar, 0 too few labels
Europe_003           patches:  144   skipped: 0 missing radar, 0 too few labels
Europe_004           patches:  144   skipped: 0 missing radar, 0 too few labels
Asia_002             patches:  144   skipped: 0 missing radar, 0 too few labels
Asia_003             patches:  144   skipped: 0 missing radar, 0 too few labels
Asia_005             patches:  144   skipped: 0 missing radar, 0 too few labels
Asia_007             patches:  112   skipped: 0 missing radar, 32 too few labels
Asia_011             patches:  144   skipped: 0 missing radar, 0 too few labels
Africa_003           patches:  144   skipped: 0 missing radar, 0 too few labels
Africa_004           patches:  120   skipped: 0 missing radar, 0 too few labels
Africa_006           patches:  144   sk